In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os
import gc

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, recall_score

In [ ]:
BASE_PATH = "/content/drive/MyDrive/MajorProject"
# Use augmented dataset for training
TRAIN_PATH = f"{BASE_PATH}/CIC_IoT_2023/Full_Dataset/augmented_train_data.csv"
TEST_PATH = f"{BASE_PATH}/CIC_IoT_2023/Full_Dataset/test_frozen.csv"

LABEL_COL = "label"
RANDOM_SEED = 42

In [4]:
print("Loading Training data")
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)

print("Loading Test data")
test_df = pd.read_csv(TEST_PATH, low_memory=False)

print("Datasets Loaded Succesfully")

Loading Training data
Loading Test data
Datasets Loaded Succesfully


In [5]:
train_df.sample(10)

,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
526274,51.182933,60897.00,7.10,164.20,28.665864,28.665864,0.0,0.0,0.0,0.0,...,572.999765,75.70,1.664303e+08,13.5,26.912896,811.867160,331098.644625,1.00,244.60,Recon-OSScan
415149,0.388456,168442.56,15.83,64.91,1083.096849,1083.096849,0.0,0.0,0.0,0.0,...,547.757075,764.10,8.337046e+07,9.5,41.172701,774.689582,316604.328646,0.95,141.55,DDoS-UDP_Fragmentation
131553,0.000000,54.00,6.00,64.00,1.022635,1.022635,0.0,0.0,0.0,0.0,...,0.000000,54.00,8.294638e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DoS-TCP_Flood
587194,1.474518,439644.70,6.00,163.10,234.507186,234.507186,0.0,0.0,0.0,0.0,...,837.451726,510.30,1.668617e+08,13.5,52.394902,1183.365036,705014.944317,1.00,244.60,DNS_Spoofing
892787,19.335536,2015.60,5.80,129.50,1.466961,1.466961,0.0,0.0,0.0,0.0,...,61.480594,109.50,2.831650e-02,5.5,14.316167,86.946690,4265.470719,0.90,38.50,BrowserHijacking
692090,38.723023,2391.30,5.30,100.00,0.939455,0.939455,0.0,0.0,0.0,0.0,...,38.483288,126.20,2.262769e-02,5.5,13.232094,54.423588,2684.977750,0.70,38.50,Recon-PortScan
487354,0.985436,532294.22,17.00,65.28,360.592234,360.592234,0.0,0.0,0.0,0.0,...,545.709574,921.34,8.337042e+07,9.5,41.534859,771.644735,313473.896620,0.95,141.55,DDoS-UDP_Fragmentation
419127,0.166600,17.02,1.24,66.20,0.887531,0.887531,0.0,0.0,0.0,0.0,...,551.323613,884.88,8.348737e+07,9.5,44.041761,779.636174,320218.135861,0.95,141.55,DDoS-ICMP_Fragmentation
319079,0.000000,0.00,47.00,64.00,26715.312102,26715.312102,0.0,0.0,0.0,0.0,...,0.000000,592.00,8.369419e+07,9.5,34.409301,0.000000,0.000000,0.00,141.55,Mirai-greeth_flood
774980,11.680848,1233.69,6.22,65.65,1.595213,1.595213,0.0,0.0,0.0,1.0,...,49.172726,73.31,8.300284e+07,9.5,12.315008,69.620097,14346.424218,0.37,141.55,DoS-HTTP_Flood


In [ ]:
print(f"Shape of training data: {train_df.shape}")
print(f"Shape of test data: {test_df.shape}")

Shape of training data: (912851, 47)
Shape of test data: (259646, 47)


### Clean data

In [7]:
train_df = train_df.dropna(subset=[LABEL_COL])
test_df = test_df.dropna(subset=[LABEL_COL])

print("Train shape after cleaning", train_df.shape)
print("Test shape after cleaning", test_df.shape)

Train shape after cleaning (912851, 47)
Test shape after cleaning (259645, 47)


### Split features and target

In [8]:
x_train = train_df.drop(columns=[LABEL_COL])
y_train = train_df[LABEL_COL]

x_test = test_df.drop(columns=[LABEL_COL])
y_test = test_df[LABEL_COL]

del train_df, test_df
gc.collect()

0

In [9]:
x_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912851 entries, 0 to 912850
Data columns (total 46 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   flow_duration    912851 non-null  float64
 1   Header_Length    912851 non-null  float64
 2   Protocol Type    912851 non-null  float64
 3   Duration         912851 non-null  float64
 4   Rate             912851 non-null  float64
 5   Srate            912851 non-null  float64
 6   Drate            912851 non-null  float64
 7   fin_flag_number  912851 non-null  float64
 8   syn_flag_number  912851 non-null  float64
 9   rst_flag_number  912851 non-null  float64
 10  psh_flag_number  912851 non-null  float64
 11  ack_flag_number  912851 non-null  float64
 12  ece_flag_number  912851 non-null  float64
 13  cwr_flag_number  912851 non-null  float64
 14  ack_count        912851 non-null  float64
 15  syn_count        912851 non-null  float64
 16  fin_count        912851 non-null  floa

In [10]:
print("NANs in x_train:", x_train.isna().sum().sum())
print("NaNs in X_test:", x_test.isna().sum().sum())
print("NaNs in y_train:", y_train.isna().sum())
print("NaNs in y_test:", y_test.isna().sum())

NANs in x_train: 0
NaNs in X_test: 0
NaNs in y_train: 0
NaNs in y_test: 0


### Encode labels

In [11]:
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

class_names = le.classes_

print("Number of classes", len(class_names))

Number of classes 34


In [15]:
# Map class names to encoded labels
class_name_to_index = {
    cls: idx for idx, cls in enumerate(le.classes_)
}

class_name_to_index

{'Backdoor_Malware': 0,
 'BenignTraffic': 1,
 'BrowserHijacking': 2,
 'CommandInjection': 3,
 'DDoS-ACK_Fragmentation': 4,
 'DDoS-HTTP_Flood': 5,
 'DDoS-ICMP_Flood': 6,
 'DDoS-ICMP_Fragmentation': 7,
 'DDoS-PSHACK_Flood': 8,
 'DDoS-RSTFINFlood': 9,
 'DDoS-SYN_Flood': 10,
 'DDoS-SlowLoris': 11,
 'DDoS-SynonymousIP_Flood': 12,
 'DDoS-TCP_Flood': 13,
 'DDoS-UDP_Flood': 14,
 'DDoS-UDP_Fragmentation': 15,
 'DNS_Spoofing': 16,
 'DictionaryBruteForce': 17,
 'DoS-HTTP_Flood': 18,
 'DoS-SYN_Flood': 19,
 'DoS-TCP_Flood': 20,
 'DoS-UDP_Flood': 21,
 'MITM-ArpSpoofing': 22,
 'Mirai-greeth_flood': 23,
 'Mirai-greip_flood': 24,
 'Mirai-udpplain': 25,
 'Recon-HostDiscovery': 26,
 'Recon-OSScan': 27,
 'Recon-PingSweep': 28,
 'Recon-PortScan': 29,
 'SqlInjection': 30,
 'Uploading_Attack': 31,
 'VulnerabilityScan': 32,
 'XSS': 33}

### Define Random Forest

In [16]:
# Class-conditional weights for hard minority classes
# These values are conservative and defensible
CLASS_WEIGHT = {
    class_name_to_index["Recon-PingSweep"]: 3.0,
    class_name_to_index["Uploading_Attack"]: 3.0,
    class_name_to_index["SqlInjection"]: 2.5,
    class_name_to_index["XSS"]: 2.5,
    class_name_to_index["Backdoor_Malware"]: 2.0,
    class_name_to_index["BrowserHijacking"]: 2.0
}

print("Encoded class weights:", CLASS_WEIGHT)

Encoded class weights: {28: 3.0, 31: 3.0, 30: 2.5, 33: 2.5, 0: 2.0, 2: 2.0}


In [17]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
    verbose=1,
    class_weight=CLASS_WEIGHT
)

### Train the model

In [18]:
print("Training Random forest on", x_train.shape[0], "rows")

rf_model.fit(x_train, y_train_enc)

Training Random forest on 912851 rows


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  3.5min
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed: 14.9min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed: 15.2min finished


RandomForestClassifier(class_weight={0: 2.0, 2: 2.0, 28: 3.0, 30: 2.5, 31: 3.0,
                                     33: 2.5},
                       n_estimators=200, n_jobs=-1, random_state=42, verbose=1)

In [19]:
print("Generating predictions...")
y_pred_enc = rf_model.predict(x_test)

Generating predictions...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    6.9s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:   21.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:   21.5s finished


In [20]:
print("Generating predictions...")
y_pred_enc = rf_model.predict(x_test)

Generating predictions...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    3.9s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:   17.8s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:   18.1s finished


In [21]:
report = classification_report(
    y_test_enc,
    y_pred_enc,
    target_names=class_names,
    digits=4
)

print(report)

                         precision    recall  f1-score   support

       Backdoor_Malware     0.9656    0.5282    0.6829       638
          BenignTraffic     0.7608    0.7447    0.7526      6000
       BrowserHijacking     0.9768    0.4687    0.6335      1167
       CommandInjection     0.9512    0.6150    0.7470      1078
 DDoS-ACK_Fragmentation     0.9981    0.9964    0.9972      7799
        DDoS-HTTP_Flood     0.9962    0.9929    0.9945      5742
        DDoS-ICMP_Flood     1.0000    0.9990    0.9995      6000
DDoS-ICMP_Fragmentation     0.9978    0.9957    0.9967      7655
      DDoS-PSHACK_Flood     0.9997    0.9997    0.9997      6000
       DDoS-RSTFINFlood     1.0000    0.9993    0.9997      6000
         DDoS-SYN_Flood     0.9985    0.9975    0.9980      6000
         DDoS-SlowLoris     0.9817    0.9972    0.9894      4672
DDoS-SynonymousIP_Flood     0.9990    0.9982    0.9986      6000
         DDoS-TCP_Flood     1.0000    0.9983    0.9992      6000
         DDoS-UDP_Flood 

In [22]:
recalls = recall_score(
    y_test_enc,
    y_pred_enc,
    average=None
)

recall_df = pd.DataFrame({
    "Class": class_names,
    "Recall": recalls
}).sort_values("Recall")

recall_df

,Class,Recall
28,Recon-PingSweep,0.229911
31,Uploading_Attack,0.412000
30,SqlInjection,0.424330
2,BrowserHijacking,0.468723
33,XSS,0.522193
0,Backdoor_Malware,0.528213
17,DictionaryBruteForce,0.586313
3,CommandInjection,0.615028
1,BenignTraffic,0.744667
29,Recon-PortScan,0.749527


In [23]:
MODEL_B_v2_OUTPUT_DIR = f"{BASE_PATH}/IDS_ModelB_v2_Outputs"
os.makedirs(MODEL_B_v2_OUTPUT_DIR, exist_ok=True)

recall_path = f"{MODEL_B_v2_OUTPUT_DIR}/model_B_v2_per_class_recall.csv"
recall_df.to_csv(recall_path, index=False)

print("✅ Per-class recall saved to:", recall_path)

✅ Per-class recall saved to: /content/drive/MyDrive/MajorProject/IDS_ModelB_v2_Outputs/model_B_v2_per_class_recall.csv


In [24]:
import joblib

joblib.dump(rf_model, f"{MODEL_B_v2_OUTPUT_DIR}/modelB_RF_v2.joblib")
joblib.dump(le, f"{MODEL_B_v2_OUTPUT_DIR}/modelB_v2_labelEncoder.joblib")

print("Model B v2 Saved succesfully")

Model B v2 Saved succesfully
